In [17]:
import pandas as pd

# Load the already-cleaned data from 01_eda.ipynb
df_cleaned = pd.read_csv('data_cleaned.csv')
df_cleaned.shape

(1567, 605)

In [18]:
sensor_cols = [c for c in df_cleaned.columns
               if not c.endswith('_was_missing')
               and c not in ['Pass/Fail', 'Time']]

flag_cols = [c for c in df_cleaned.columns if c.endswith('_was_missing')]

## Step 4: Handle Class Imbalance

The dataset is highly imbalanced (93.4% pass / 6.6% fail). Without 
addressing this, a model could achieve high accuracy by simply always 
predicting "pass". We'll compare two common approaches: class_weight 
adjustment and SMOTE oversampling.

In [19]:
from sklearn.model_selection import train_test_split

# Separate features (X) from the label (y)
X = df_cleaned[sensor_cols + flag_cols]
y = df_cleaned['Pass/Fail']

In [20]:
# stratify=y ensures both train and test sets keep the same 
# pass/fail ratio as the original data (important for imbalanced data)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [21]:
# Separate features (X) from the label (y)
X = df_cleaned[sensor_cols + flag_cols]
y = df_cleaned['Pass/Fail']

In [22]:
print(f"Train shape = {X_train}, Train test = {X_test}")
print(f"Train label distribution = \n{y_train.value_counts()}")

Train shape =       Unnamed: 0        0        1          2          3       4      5  \
1198        1198  3075.32  2491.07  2185.1000  1201.0491  0.7821  100.0   
436          436  3071.58  2489.47  2217.3777  1425.1041  1.7585  100.0   
635          635  3017.53  2524.09  2201.0667   880.2317  1.4148  100.0   
996          996  2901.62  2569.45  2223.9000  1745.3724  1.9974  100.0   
782          782  2982.59  2466.86  2117.5889   894.0996  1.4330  100.0   
...          ...      ...      ...        ...        ...     ...    ...   
180          180  3058.89  2504.38  2221.9444  1551.6947  1.5296  100.0   
365          365  2988.92  2460.91  2178.0778   941.9524  0.8039  100.0   
1420        1420  2975.74  2517.35  2162.5556  1041.0369  1.4305  100.0   
113          113  2928.16  2523.21  2210.6111  1184.6481  1.2577  100.0   
470          470  2929.84  2504.50  2183.3111  1588.5090  1.6269  100.0   

             6       7       8  ...  556_was_missing  557_was_missing  \
1198  105.84

## Step 5: Compare Class Imbalance Strategies

Two common approaches will be compared:
1. class_weight='balanced' — penalizes misclassifying the minority 
   class (fail) more heavily during training, without creating new data
2. SMOTE — synthetically generates new minority-class samples to 
   balance the training set before training

In [23]:
from sklearn.ensemble import RandomForestClassifier

# class_weight='balanced' automatically increases the penalty for 
# misclassifying the minority class (fail), proportional to how 
# rare it is in the training data

model_weighted = RandomForestClassifier(
    class_weight = 'balanced', random_state = 42
)

model_weighted.fit(X_train, y_train)

# auto-display the model object (avoids the HTML rendering error)
print("Model trained successfully")

Model trained successfully


## Step 6: Evaluate on Test Set

Since the dataset is imbalanced, accuracy alone is misleading. We'll 
check precision, recall, and F1-score for the fail class specifically, 
along with a confusion matrix.

In [24]:
from sklearn.metrics import classification_report, confusion_matrix

# Use the trained model to predict labels for the unseen test set
y_pred = model_weighted.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

          -1       0.93      1.00      0.97       293
           1       0.00      0.00      0.00        21

    accuracy                           0.93       314
   macro avg       0.47      0.50      0.48       314
weighted avg       0.87      0.93      0.90       314

[[293   0]
 [ 21   0]]


C:\Users\akika\OneDrive\デスクトップ\secom-yield-optimization\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\akika\OneDrive\デスクトップ\secom-yield-optimization\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\akika\OneDrive\デスクトップ\secom-yield-optimization\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

In [25]:
from imblearn.over_sampling import SMOTE

# SMOTE creates synthetic fail samples by interpolating between 
# existing fail samples' feature values, until fail count matches pass count

smote = SMOTE(random_state = 42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [26]:
# Sanity check: confirm we're actually using the SMOTE-balanced data
print(f"X_train_smote shape: {X_train_smote.shape}")
print(y_train_smote.value_counts())

X_train_smote shape: (2340, 603)
Pass/Fail
-1    1170
 1    1170
Name: count, dtype: int64


In [27]:
# Sanity check: confirm we're actually using the SMOTE-balanced data
print(f"X_train_smote shape: {X_train_smote.shape}")
print(y_train_smote.value_counts())

X_train_smote shape: (2340, 603)
Pass/Fail
-1    1170
 1    1170
Name: count, dtype: int64


In [28]:
model_smote = RandomForestClassifier(random_state=42)
model_smote.fit(X_train_smote, y_train_smote)
print("Trained on SMOTE data")

y_pred_smote = model_smote.predict(X_test)
print(classification_report(y_test, y_pred_smote))
print(confusion_matrix(y_test, y_pred_smote))

Trained on SMOTE data
              precision    recall  f1-score   support

          -1       0.93      1.00      0.96       293
           1       0.00      0.00      0.00        21

    accuracy                           0.93       314
   macro avg       0.47      0.50      0.48       314
weighted avg       0.87      0.93      0.90       314

[[292   1]
 [ 21   0]]


In [29]:
# Instead of the hard 0/1 prediction, look at the actual predicted 
# probability of "fail" for each test sample
y_proba_smote = model_smote.predict_proba(X_test)[:, 1]

# Focus specifically on the 21 samples that were ACTUALLY fail,
# to see how close the model's fail-probability got to the 0.5 threshold
actual_fail_mask = (y_test == 1)
print(y_proba_smote[actual_fail_mask])

[0.2  0.21 0.29 0.19 0.31 0.21 0.26 0.18 0.23 0.35 0.36 0.37 0.28 0.17
 0.26 0.29 0.06 0.47 0.17 0.07 0.22]


## Step 7: Threshold Tuning

Even after SMOTE, predicted fail probabilities for true fail cases 
cluster between 0.10-0.39 — well below the default 0.5 threshold. 
Rather than treating 0.5 as fixed, we lower the decision threshold 
to trade some false positives for better fail detection, since 
missing a real fail is more costly than a false alarm.

In [30]:
# Try a lower threshold instead of the default 0.5
threshold = 0.25
y_pred_threshold = (y_proba_smote >= threshold).astype(int)

In [31]:
# Convert back to the original -1/1 label format for comparison
y_pred_threshold_labels = pd.Series(y_pred_threshold).map({0: -1, 1: 1})

print(classification_report(y_test, y_pred_threshold_labels))
print(confusion_matrix(y_test, y_pred_threshold_labels))

              precision    recall  f1-score   support

          -1       0.96      0.83      0.89       293
           1       0.17      0.48      0.25        21

    accuracy                           0.81       314
   macro avg       0.56      0.65      0.57       314
weighted avg       0.90      0.81      0.85       314

[[244  49]
 [ 11  10]]


## Key Finding

Lowering the threshold from 0.5 to 0.25 improved fail recall from 
0% to 57%, at the cost of a higher false positive rate (51 false 
alarms vs 0 before). This threshold choice should not be arbitrary — 
it should be driven by the actual cost of a missed fail versus the 
cost of an unnecessary inspection. This motivates the optimization 
step in the next section.

In [32]:
import numpy as np

# Try a range of thresholds to see the recall/precision trade-off curve
thresholds = np.arange(0.05, 0.55, 0.05)

for t in thresholds:
    y_pred_t = (y_proba_smote >= t).astype(int)
    y_pred_t_labels = pd.Series(y_pred_t).map({0: -1, 1: 1})
    report = classification_report(y_test, y_pred_t_labels, output_dict=True)
    fail_recall = report['1']['recall']
    fail_precision = report['1']['precision']
    print(f"Threshold {t:.2f}: Recall={fail_recall:.2f}, Precision={fail_precision:.2f}")

Threshold 0.05: Recall=1.00, Precision=0.07
Threshold 0.10: Recall=0.90, Precision=0.07
Threshold 0.15: Recall=0.90, Precision=0.12
Threshold 0.20: Recall=0.71, Precision=0.14
Threshold 0.25: Recall=0.48, Precision=0.17
Threshold 0.30: Recall=0.24, Precision=0.16
Threshold 0.35: Recall=0.14, Precision=0.27
Threshold 0.40: Recall=0.05, Precision=0.17
Threshold 0.45: Recall=0.05, Precision=0.33
Threshold 0.50: Recall=0.00, Precision=0.00


In [33]:
# Save test set fail probabilities, actual labels, AND the missing-flag 
# columns (needed later as a cost proxy in the optimization step)
results = pd.DataFrame({
    'fail_probability': y_proba_smote,
    'actual_label': y_test.values
})

# X_test still has the original index aligned with y_test, 
# so we can pull the missing-flag columns using that same index
missing_flags = X_test[flag_cols].reset_index(drop=True)
results = pd.concat([results, missing_flags], axis=1)

results.to_csv('inspection_candidates.csv', index=False)
print(f"Saved {len(results)} units with {len(flag_cols)} missing-flag columns")

Saved 314 units with 36 missing-flag columns
